<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/02_Taller_Espacios_Vectoriales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: espacios vectoriales</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `05_Espacios_Vectoriales.ipynb` con **datos y ejercicios distintos** a los de las notas de clase. Usaremos el corpus **Reuters-21578** de NLTK: noticias económicas cortas clasificadas por tema, con una división oficial en entrenamiento y prueba.

Enfocamos:

- las medidas de similitud y TF-IDF implementadas **desde cero**,
- la **evaluación cuantitativa** de un buscador (precisión@k),
- la clasificación por **vecinos más cercanos** y por **centroides** en el espacio vectorial,
- el efecto del **tamaño de la ventana** en los vectores de palabras por co-ocurrencia,
- la visualización e interpretación de tópicos latentes.

Resuelve cada ejercicio en la celda indicada. Cuando se pida una interpretación, respóndela en una celda markdown.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Medidas de similitud desde cero</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. TF-IDF sublineal desde cero</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Un buscador y su evaluación</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Clasificación en el espacio vectorial</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Vectores de palabras y el tamaño de la ventana</span>](#ejercicio-5)
- [<span style="color: #005C99;">Ejercicio 6. Tópicos latentes y visualización</span>](#ejercicio-6)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib seaborn scikit-learn scipy nltk

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

import nltk
from nltk.corpus import reuters

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)

nltk.download("reuters", quiet=True)

# Usamos noticias con una sola categoría, de 8 temas
CATEGORIAS = ["crude", "trade", "money-fx", "interest", "money-supply", "ship", "sugar", "coffee"]
docs = [f for f in reuters.fileids() if len(reuters.categories(f)) == 1 and reuters.categories(f)[0] in CATEGORIAS]

train_ids = [f for f in docs if f.startswith("training/")]
test_ids = [f for f in docs if f.startswith("test/")]

X_train_txt = [reuters.raw(f) for f in train_ids]
X_test_txt = [reuters.raw(f) for f in test_ids]
y_train = np.array([reuters.categories(f)[0] for f in train_ids])
y_test = np.array([reuters.categories(f)[0] for f in test_ids])

print(f"Entrenamiento: {len(train_ids)} | Prueba: {len(test_ids)}")
print(pd.Series(y_train).value_counts().to_dict())
print("\nEjemplo:\n", X_train_txt[0][:400])

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Medidas de similitud desde cero</h2>

**Objetivo:** implementar las medidas de similitud con NumPy y comprobar su relación cuando los vectores están normalizados.

1. Implementa `producto_punto(a, b)`, `norma(a)`, `similitud_coseno(a, b)` y `distancia_euclidiana(a, b)` usando solo operaciones de NumPy (sin `np.linalg.norm` ni funciones de scikit-learn).
2. Verifícalas contra `cosine_similarity` y `euclidean_distances` de scikit-learn con los vectores de conteo de dos noticias de entrenamiento (`CountVectorizer`).
3. Normaliza los dos vectores a norma 1 y comprueba numéricamente la identidad $\|\mathbf{a} - \mathbf{b}\|^2 = 2 - 2\cos(\mathbf{a}, \mathbf{b})$. Demuéstrala algebraicamente en una celda markdown.
4. ¿Qué consecuencia práctica tiene esta identidad para un buscador que usa vectores TF-IDF normalizados?

In [ ]:
# Escribe tu código aquí
def producto_punto(a, b):
    pass


def norma(a):
    pass


def similitud_coseno(a, b):
    pass


def distancia_euclidiana(a, b):
    pass

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. TF-IDF sublineal desde cero</h2>

**Objetivo:** implementar TF-IDF sobre matrices dispersas, incluida la variante sublineal, y validarla contra scikit-learn.

En documentos largos, que una palabra aparezca 20 veces no la hace 20 veces más importante que si aparece una vez. La variante **sublineal** amortigua la frecuencia con un logaritmo:

$$\text{tf}_{\text{sub}}(t, d) = \begin{cases} 1 + \log \text{tf}(t, d) & \text{si } \text{tf}(t, d) > 0 \\ 0 & \text{en otro caso} \end{cases}$$

1. Con `CountVectorizer(stop_words="english")` obtén la matriz de conteos de entrenamiento `C_train` (dispersa).
2. Implementa `ajustar_idf(C)` que devuelva el vector IDF suavizado de scikit-learn: $\ln\frac{1 + N}{1 + \text{df}(t)} + 1$.
3. Implementa `transformar_tfidf(C, idf, sublineal=True)` que devuelva una matriz **dispersa** normalizada por filas (norma L2). Pista: en una matriz `csr`, `C.data` contiene solo los valores no nulos, así que puedes aplicar el logaritmo sobre `C.data` directamente.
4. Verifica que tu resultado coincide con `TfidfVectorizer(stop_words="english", sublinear_tf=True)`.
5. Para una noticia larga, compara los 8 términos con mayor peso con TF lineal y con TF sublineal. ¿Qué cambia?

In [ ]:
# Escribe tu código aquí
def ajustar_idf(C):
    pass


def transformar_tfidf(C, idf, sublineal=True):
    pass

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Un buscador y su evaluación</h2>

**Objetivo:** construir un buscador por similitud coseno y comparar representaciones con una métrica cuantitativa (precisión@k).

No tenemos juicios de relevancia, pero podemos usar las categorías como aproximación: **un resultado es relevante si tiene la misma categoría que la consulta**. Usaremos cada noticia de prueba como consulta contra la colección de entrenamiento.

$$\text{precisión@}k = \frac{\#\{\text{resultados relevantes entre los primeros } k\}}{k}$$

1. Implementa `precision_at_k(Q, D, y_q, y_d, k)` que, dadas las matrices de consultas `Q` y de documentos `D`, calcule la precisión@k promedio sobre todas las consultas (usa `cosine_similarity`, que acepta matrices dispersas).
2. Compara la precisión@10 de estas representaciones (ajustadas solo con entrenamiento):
   - conteos crudos (`CountVectorizer()`),
   - conteos sin stopwords,
   - TF-IDF sin stopwords,
   - TF-IDF sin stopwords y sublineal,
   - TF-IDF con unigramas y bigramas (`ngram_range=(1, 2)`, `min_df=2`).
3. Presenta los resultados en una tabla y una gráfica de barras.
4. Implementa `buscar(consulta, k=5)` con la mejor representación y pruébala con al menos tres consultas propias.
5. ¿Qué representación funciona mejor y por qué?

In [ ]:
# Escribe tu código aquí
def precision_at_k(Q, D, y_q, y_d, k=10):
    pass

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Clasificación en el espacio vectorial</h2>

**Objetivo:** usar la geometría del espacio TF-IDF para clasificar documentos con vecinos más cercanos y con centroides, y compararlos con un clasificador lineal.

Si documentos del mismo tema quedan cerca en el espacio vectorial, podemos clasificar una noticia nueva mirando **dónde cae**:

- **k vecinos más cercanos (kNN):** asigna la categoría mayoritaria entre las $k$ noticias de entrenamiento más similares (similitud coseno).
- **Centroides (Rocchio):** calcula el vector promedio de cada categoría y asigna la categoría del centroide más similar.

1. Con la representación TF-IDF sublineal sin stopwords (`X_tr`, `X_te`), implementa **tú mismo** el clasificador por centroides: calcula los centroides, normalízalos y asigna la categoría con mayor similitud coseno. Reporta la exactitud.
2. Entrena `KNeighborsClassifier(metric="cosine")` con $k \in \{1, 5, 15\}$ y reporta la exactitud de cada uno.
3. Como referencia, entrena `LogisticRegression(max_iter=2000)`.
4. Muestra la matriz de confusión del mejor modelo. ¿Qué categorías se confunden y por qué crees que ocurre?

In [ ]:
# Escribe tu código aquí
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True)
X_tr = vec.fit_transform(X_train_txt)
X_te = vec.transform(X_test_txt)

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Vectores de palabras y el tamaño de la ventana</h2>

**Objetivo:** construir vectores de palabras PPMI + SVD y analizar cómo el tamaño de la ventana de contexto cambia el tipo de similitud capturada.

1. Tokeniza todas las noticias de **entrenamiento** (usa `CountVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{2,}\b").build_analyzer()`) y quédate con las 3 000 palabras más frecuentes.
2. Implementa `vectores_ppmi_svd(tokens_docs, vocab, ventana, dim=100)` que construya la matriz de co-ocurrencia (dispersa), aplique PPMI, reduzca con `TruncatedSVD` y devuelva los vectores normalizados.
3. Construye dos modelos: `ventana=2` y `ventana=10`.
4. Para las palabras `oil`, `coffee`, `bank`, `rose` y `tonnes`, muestra los 6 vecinos más cercanos en cada modelo.
5. ¿Qué diferencia observas entre las vecindades de ventana pequeña y grande? Relaciónalo con la diferencia entre similitud **sintáctica / funcional** y similitud **temática**.

In [ ]:
# Escribe tu código aquí
def vectores_ppmi_svd(tokens_docs, vocab, ventana, dim=100):
    pass

---

<a id="ejercicio-6"></a>
<h2 style="color: #007ACC;">Ejercicio 6. Tópicos latentes y visualización</h2>

**Objetivo:** usar LSA para descubrir tópicos latentes y visualizar las noticias en dos dimensiones.

1. Aplica `TruncatedSVD(n_components=50)` sobre `X_tr` (TF-IDF del ejercicio 4) y muestra los 8 términos con mayor peso de las primeras 6 componentes. Intenta ponerle un nombre a cada "tópico".
2. Proyecta las noticias de **prueba** a esas 50 dimensiones (con el SVD ajustado en entrenamiento) y luego a 2D con PCA. Grafica coloreando por categoría.
3. Entrena una `LogisticRegression` sobre las 50 dimensiones de LSA y compara su exactitud con la del modelo sobre TF-IDF completo del ejercicio 4. ¿Cuánta información se pierde al pasar de miles de dimensiones a 50?

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Lane, H., Howard, C. & Hapke, H. (2019). *Natural Language Processing in Action*. Manning.
- Documentación de scikit-learn: *Working With Text Data*. [scikit-learn.org](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)
- Manning, C. D., Raghavan, P. & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press. [nlp.stanford.edu/IR-book](https://nlp.stanford.edu/IR-book/)